# Calculating regression coefficients for individual conversations in a given dataset

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.linalg import lstsq

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import statsmodels.formula.api as smf
import warnings;warnings.filterwarnings('ignore')

In [2]:
CORPUS_NAME = 'CORAAL'

In [3]:
DATA_PATH = '../data/ckpts/rosen'

OUTPUTS_PATH = 'reports'
if not os.path.exists(OUTPUTS_PATH):
    os.mkdir(OUTPUTS_PATH)

In [4]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [5]:
fs = [f for f in grab_all_files(DATA_PATH) if f.split('/')[-1].startswith(CORPUS_NAME)]
len(fs)

1

## Processing individual datasets

In [6]:
# formula = '{} ~ nx + ny + self'.format(COEF_VAR)

In [7]:
COEF_VAR = 'I'

In [8]:
# param_list = ['Intercept', 'nx', 'ny', 'self']
param_list = ['nx', 'ny', 'self']

In [9]:
df_scale, df_regression = [], []

In [10]:
for f in fs:
    # print(f.split('/')[-1])
    table = pq.ParquetFile(f)

    df = table.read(columns=[
        'corpus', 'conversation_id', 'x_speaker', 'y_speaker',
        COEF_VAR,
        # 'x_turn_id', 'y_turn_id', 'AVAR',
        'nx', 'ny']).to_pandas()

    df['self'] = (df['x_speaker'] == df['y_speaker']).astype(int)

    df['Intercept'] = 1.

    for corpus in tqdm(df['corpus'].unique()):
        sub = df.loc[df['corpus'].isin([corpus])]

        groups = sub.groupby(by=['conversation_id'])
        for labels, dfi in tqdm(groups):
            params, _, _, _ = lstsq(dfi[param_list].values, dfi[COEF_VAR].values)

            df_regression += [
                {
                    'corpus': corpus,
                    'cid': labels[0],
                    'param': name,
                    'beta': param,
                    'k': len(dfi)
                }
            for name, param in list(zip(param_list,params))]

df_regression = pd.DataFrame(df_regression)


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/271 [00:00<?, ?it/s]

In [11]:
if not os.path.exists(os.path.join(OUTPUTS_PATH,'all.csv')):
    df_regression.to_csv(
        os.path.join(OUTPUTS_PATH,'all.csv'),
        index=False,
        encoding='utf-8'
    )

else:
    df_regression.to_csv(
        os.path.join(OUTPUTS_PATH,'all.csv'),
        index=False,
        header=False,
        encoding='utf-8',
        mode='a'
    )

## Corpus level analysis of results

In [12]:
split_by = ['corpus', 'param']

In [13]:
df0 = df_regression[split_by+['beta']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['beta']].groupby(by=split_by).agg('std').reset_index()['beta'].values

df0['k'] = df_regression[split_by+['cid']].groupby(by=split_by).agg('count').reset_index()['cid'].values
# df0['k'] = df_regression[split_by+['k']].groupby(by=split_by).agg('sum').reset_index()['k'].values

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [14]:
df0['t'] = df0['beta'] / df0['se']

In [15]:
df0.head(n=20)

,corpus,param,beta,std,k,se,t
0,CORAAL,nx,0.256937,0.025820,271,0.001568,163.816106
1,CORAAL,ny,-0.039053,0.009165,271,0.000557,-70.147269
2,CORAAL,self,-0.253563,0.205840,271,0.012504,-20.278766


In [16]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, CORPUS_NAME+'.csv'),
    index=False, encoding='utf-8'
)